## Step1 download dublinbike condition through API of JCD and keep the data in a json file saved in local harddisk  which is not recommended but if you don't set the database in the begaining.

In [1]:
import json
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path
import requests
import sys
import os


sys.path.append(os.getcwd())
try:
    import config # api info should not be shown in public
except ImportError:
    sys.path.append(os.path.dirname(os.getcwd())) # to find the folder in parent dir
    import config

output_dir = Path("data/dublinbike_status")
output_dir.mkdir(parents=True, exist_ok=True)

def fetch_and_save_once():
    ts = datetime.now(timezone.utc)
    ts_str = ts.strftime("%Y%m%dT%H%M%SZ")
    filename = output_dir / f"station_status_{ts_str}.json"
    try:
        print(f"[{ts.strftime('%Y-%m-%d %H:%M:%S')} UTC] Fetching from JCD API", flush=True)
        resp = requests.get(config.BIKE_STATUS_URL, timeout=20) # set timeout 
        print(f"[{ts.strftime('%Y-%m-%d %H:%M:%S')} UTC] Status Code: {resp.status_code}", flush=True)
        resp.raise_for_status() # raise exception for HTTP errors
        data = resp.json() # may raise JSONDecodeError if response is not valid JSON
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False) # save info in UTF-8 format
        print(f"[{ts.strftime('%Y-%m-%d %H:%M:%S')} UTC] Saved: {filename}", flush=True)
        print(f"[{ts.strftime('%Y-%m-%d %H:%M:%S')} UTC] Next update in 5 minutes...", flush=True)
    except Exception as e:
        print(f"[{ts.strftime('%Y-%m-%d %H:%M:%S')} UTC] Error: {e}; will retry next cycle", flush=True)


max_hours = 48 
end_time = datetime.now(timezone.utc) + timedelta(hours=max_hours) #keep fetching for 48 hours, then stop,or manually stop by clicking button in jupyter notebook

while datetime.now(timezone.utc) < end_time:
    fetch_and_save_once()
    time.sleep(300) #sleep for 5 minutes
print("done: reached 48h timeout")


[2026-02-12 21:30:14 UTC] Fetching from JCD API
[2026-02-12 21:30:14 UTC] Status Code: 200
[2026-02-12 21:30:14 UTC] Saved: data/dublinbike_status/station_status_20260212T213014Z.json
[2026-02-12 21:30:14 UTC] Next update in 5 minutes...


KeyboardInterrupt: 

# Step 2 initialize the database confirm the connection with the database. 

In [ ]:
import os
import sys
import sqlalchemy as sqla

# check  config status
sys.path.append(os.getcwd())
try:
    import config
    print("config checked", flush=True)
except ImportError:
    print("Error: config.py not found,pls make one at first.")
    raise

DB_NAME = "COMP30830_SW"

# connect to MySQL and create database if not exists
engine = sqla.create_engine(
    f"mysql+pymysql://{config.DB_USER}:{config.DB_PASSWORD}@{config.DB_HOST}/"
    f"?charset=utf8mb4"
)

with engine.connect() as conn:
    exists = conn.execute(
        sqla.text("SHOW DATABASES LIKE :name"),
        {"name": DB_NAME}
    ).fetchone() is not None

    if exists:
        print(f"Database already exists: {DB_NAME}", flush=True)
    else:
        conn.execute(sqla.text(f"CREATE DATABASE `{DB_NAME}` DEFAULT CHARACTER SET utf8mb4"))
        conn.commit()
        print(f"Database created: {DB_NAME}", flush=True)


#  Step 3 pull json file into database and check the describe of the data.

In [2]:
import json
from pathlib import Path
from datetime import datetime, timezone
import sqlalchemy as sqla
import config
DB_NAME = "COMP30830_SW"

# confirm connection to the new database
engine = sqla.create_engine(
    f"mysql+pymysql://{config.DB_USER}:{config.DB_PASSWORD}@{config.DB_HOST}/{DB_NAME}"
    f"?charset=utf8mb4"
)

# Create tables (station + availability)
create_station_sql = """
CREATE TABLE IF NOT EXISTS station (
    number INT PRIMARY KEY,
    contract_name VARCHAR(64),
    name VARCHAR(128),
    address VARCHAR(256),
    lat DECIMAL(9,6),
    lng DECIMAL(9,6),
    banking TINYINT(1),
    bonus TINYINT(1),
    bike_stands INT
);
"""

create_availability_sql = """
CREATE TABLE IF NOT EXISTS availability (
    id BIGINT AUTO_INCREMENT PRIMARY KEY,
    number INT NOT NULL,
    available_bike_stands INT,
    available_bikes INT,
    status VARCHAR(16),
    last_update DATETIME,
    INDEX idx_number_time (number, last_update),
    CONSTRAINT fk_station_number
        FOREIGN KEY (number) REFERENCES station(number)
        ON DELETE CASCADE
);
"""

with engine.begin() as conn: #use SQLAlchemy's transaction context to ensure atomicity
    conn.execute(sqla.text(create_station_sql))
    conn.execute(sqla.text(create_availability_sql))

print("Table structure confirmed", flush=True)

# Read local JSON files 
input_path = Path("data/dublinbike_status")

json_files = []
if input_path.is_dir():
    json_files = sorted(input_path.glob("station_status_*.json"))# only read files matching the pattern, and sort by name (timestamp)
else:
    json_files = [input_path] # if it's a single file, just put it in a list

print(f" Found {len(json_files)} files", flush=True)

# Insert data
station_rows = {} #use dict to deduplicate station info by number,should keep 115 records for the real stops
availability_rows = [] #keep all availability records, should be 115 * number of files
for fp in json_files:
    data = json.loads(fp.read_text(encoding="utf-8"))# read Json file by python function

    for s in data:
        number = s.get("number")
        if number is None:# drop the record if number is missing, as it's the primary key for station and foreign key for availability
            continue
        # station only keeps one record ( number is key)
        station_rows[number] = {
            "number": number,
            "contract_name": s.get("contract_name"),
            "name": s.get("name"),
            "address": s.get("address"),
            "lat": (s.get("position") or {}).get("lat"),
            "lng": (s.get("position") or {}).get("lng"),
            "banking": 1 if s.get("banking") else 0, # will get deleted for all records is false
            "bonus": 1 if s.get("bonus") else 0, # same to banking
            "bike_stands": s.get("bike_stands"),
        }

        # availability keeps every record
        last_update_ms = s.get("last_update")
        last_update_dt = None
        if last_update_ms:
            # standardize to UTC datetime, as the API returns timestamp in milliseconds, for later machine learning use
            last_update_dt = datetime.fromtimestamp(last_update_ms / 1000, tz=timezone.utc)
       
        availability_rows.append({
            "number": number,
            "available_bike_stands": s.get("available_bike_stands"),
            "available_bikes": s.get("available_bikes"),
            "status": s.get("status"),
            "last_update": last_update_dt,
        })

print(f"station has {len(station_rows)} records", flush=True)
print(f"availability has {len(availability_rows)} records", flush=True)

# Batch write (upsert for station to save time and resources; direct insert for availability)
station_insert = sqla.text("""
INSERT INTO station (number, contract_name, name, address, lat, lng, banking, bonus, bike_stands)
VALUES (:number, :contract_name, :name, :address, :lat, :lng, :banking, :bonus, :bike_stands)
ON DUPLICATE KEY UPDATE
    contract_name = VALUES(contract_name),
    name = VALUES(name),
    address = VALUES(address),
    lat = VALUES(lat),
    lng = VALUES(lng),
    banking = VALUES(banking),
    bonus = VALUES(bonus),
    bike_stands = VALUES(bike_stands);
""")

availability_insert = sqla.text("""
INSERT INTO availability (number, available_bike_stands, available_bikes, status, last_update)
VALUES (:number, :available_bike_stands, :available_bikes, :status, :last_update);
""")

with engine.begin() as conn:
    conn.execute(station_insert, list(station_rows.values()))
    conn.execute(availability_insert, availability_rows)

print("Data import completed", flush=True)


Table structure confirmed
 Found 582 files
station has 115 records
availability has 66350 records
Data import completed


#  Step 3.5 pull in data into database by API instead,so no need keep json file.

In [ ]:
import requests
from datetime import datetime, timezone
import sqlalchemy as sqla

# pull in data from API and insert into database.
DB_NAME = "COMP30830_SW"
engine = sqla.create_engine(
    f"mysql+pymysql://{config.DB_USER}:{config.DB_PASSWORD}@{config.DB_HOST}/{DB_NAME}"
    f"?charset=utf8mb4"
)

# Execute one pull + insert
resp = requests.get(config.BIKE_STATUS_URL, timeout=20)
resp.raise_for_status()
data = resp.json()  # Here we assume the top level is a list

station_rows = {}
availability_rows = []

for s in data:
    number = s.get("number") #primary key for station and foreign key for availability, so if missing just drop the record
    if number is None:
        continue

    station_rows[number] = {
        "number": number,
        "contract_name": s.get("contract_name"),
        "name": s.get("name"),
        "address": s.get("address"),
        "lat": (s.get("position") or {}).get("lat"),
        "lng": (s.get("position") or {}).get("lng"),
        "banking": 1 if s.get("banking") else 0,
        "bonus": 1 if s.get("bonus") else 0,
        "bike_stands": s.get("bike_stands"),
    }

    last_update_ms = s.get("last_update")
    last_update_dt = None
    if last_update_ms:
        last_update_dt = datetime.fromtimestamp(last_update_ms / 1000, tz=timezone.utc)

    availability_rows.append({
        "number": number,
        "available_bike_stands": s.get("available_bike_stands"),
        "available_bikes": s.get("available_bikes"),
        "status": s.get("status"),
        "last_update": last_update_dt,
    })

station_insert = sqla.text("""
INSERT INTO station (number, contract_name, name, address, lat, lng, banking, bonus, bike_stands)
VALUES (:number, :contract_name, :name, :address, :lat, :lng, :banking, :bonus, :bike_stands)
ON DUPLICATE KEY UPDATE
    contract_name = VALUES(contract_name),
    name = VALUES(name),
    address = VALUES(address),
    lat = VALUES(lat),
    lng = VALUES(lng),
    banking = VALUES(banking),
    bonus = VALUES(bonus),
    bike_stands = VALUES(bike_stands);
""")

availability_insert = sqla.text("""
INSERT INTO availability (number, available_bike_stands, available_bikes, status, last_update)
VALUES (:number, :available_bike_stands, :available_bikes, :status, :last_update);
""")

with engine.begin() as conn:
    conn.execute(station_insert, list(station_rows.values()))
    conn.execute(availability_insert, availability_rows)

print(f"API import completed: stations={len(station_rows)}, availability={len(availability_rows)}", flush=True)
